In [ ]:
!kaggle datasets download -d punitkashyap2007/virgo-files

In [ ]:
import os

ZIP_PATH = "./virgo-files.zip"

print("ZIP exists:", os.path.exists(ZIP_PATH))
print("ZIP size  :", round(os.path.getsize(ZIP_PATH) / 1024**3, 2), "GB")

In [ ]:
!mkdir -p virgo_data
!unzip -q -o virgo-files.zip -d virgo_data

In [ ]:
import os

for root, _, files in os.walk("./virgo_data"):
    for file in files:
        if file.endswith(".bin") or file.endswith(".json"):
            print(os.path.join(root, file))

In [ ]:
def find_file(filename, search_path="./virgo_data"):
    for root, _, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)

    return None

TOKENIZER_PATH = find_file("virgo_tokenizer.json")
TRAIN_BIN = find_file("train.bin")
VAL_BIN = find_file("val.bin")

print("Tokenizer :", TOKENIZER_PATH)
print("Train BIN :", TRAIN_BIN)
print("Val BIN   :", VAL_BIN)

In [ ]:
assert TOKENIZER_PATH is not None, "❌ virgo_tokenizer.json not found"
assert TRAIN_BIN is not None, "❌ train.bin not found"
assert VAL_BIN is not None, "❌ val.bin not found"

assert os.path.exists(TOKENIZER_PATH)
assert os.path.exists(TRAIN_BIN)
assert os.path.exists(VAL_BIN)

print("✅ Virgo tokenizer found")
print("✅ Virgo train.bin found")
print("✅ Virgo val.bin found")

In [ ]:
train_size = os.path.getsize(TRAIN_BIN) / 1024**3
val_size = os.path.getsize(VAL_BIN) / 1024**3

print(f"train.bin : {train_size:.2f} GB")
print(f"val.bin   : {val_size:.2f} GB")

In [ ]:
os.remove("./virgo-files.zip")

print("✅ ZIP deleted")
print("Extracted Virgo dataset preserved")

In [2]:
import os
import math
import time
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer

In [3]:
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)
    print("VRAM            :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch version : 2.10.0+cpu
CUDA available  : False


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
TOKENIZER_PATH = "./virgo_tokenizer.json"
TRAIN_BIN = "./train.bin"
VAL_BIN = "./val.bin"

assert os.path.exists(TOKENIZER_PATH), "Tokenizer not found"
assert os.path.exists(TRAIN_BIN), "train.bin not found"
assert os.path.exists(VAL_BIN), "val.bin not found"

print("✅ All Virgo files found")

AssertionError: Tokenizer not found

In [6]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

vocab_size = tokenizer.get_vocab_size()

print("Vocabulary size:", vocab_size)

for token in ["<pad>", "<unk>", "<bos>", "<eos>", "<newline>", "<tab>"]:
    print(f"{token:<10} → {tokenizer.token_to_id(token)}")

Exception: No such file or directory (os error 2)

In [ ]:
train_tokens = os.path.getsize(TRAIN_BIN) // np.dtype(np.uint16).itemsize
val_tokens = os.path.getsize(VAL_BIN) // np.dtype(np.uint16).itemsize

print(f"Train tokens : {train_tokens:,} ({train_tokens / 1e9:.3f}B)")
print(f"Val tokens   : {val_tokens:,} ({val_tokens / 1e6:.2f}M)")
print(f"Total tokens : {train_tokens + val_tokens:,}")

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

print("Train memmap:", train_data.shape)
print("Val memmap  :", val_data.shape)

print("Train token range:", int(train_data[:100000].min()), "→", int(train_data[:100000].max()))
print("Val token range  :", int(val_data[:100000].min()), "→", int(val_data[:100000].max()))

assert train_data[:100000].max() < vocab_size
assert val_data[:100000].max() < vocab_size

print("✅ BIN token IDs valid")

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super(RotaryEmbedding, self).__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack((Q_even * cos - Q_odd * sin, Q_even * sin + Q_odd * cos), dim=-1).flatten(-2)
        K = torch.stack((K_even * cos - K_odd * sin, K_even * sin + K_odd * cos), dim=-1).flatten(-2)

        return Q, K

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [ ]:
class VirgoBase(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoBase, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [ ]:
d_model = 768
num_heads = 12
num_layers = 12
d_ff = 3072
max_seq_length = 1024
dropout = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
model = VirgoBase(vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Model size           : {total_params / 1e6:.2f}M")

In [ ]:
model.eval()

test_input = torch.randint(0, vocab_size, (2, 128), device=device)

with torch.no_grad():
    test_logits = model(test_input)

print("Input shape :", test_input.shape)
print("Logits shape:", test_logits.shape)

assert test_logits.shape == (2, 128, vocab_size)

print("✅ Virgo Base forward pass successful")

del test_input
del test_logits

torch.cuda.empty_cache()

In [ ]:
model.eval()

input_a = torch.randint(0, vocab_size, (1, 32), device=device)
input_b = input_a.clone()

input_b[:, 16:] = torch.randint(0, vocab_size, (1, 16), device=device)

with torch.no_grad():
    logits_a = model(input_a)
    logits_b = model(input_b)

past_diff = (logits_a[:, :16] - logits_b[:, :16]).abs().max().item()

print(f"Maximum past-logit difference: {past_diff:.10f}")

assert past_diff < 1e-5, "Causal attention is leaking future information"

print("✅ Virgo causal attention test passed")

del input_a
del input_b
del logits_a
del logits_b

torch.cuda.empty_cache()

In [ ]:
def get_batch(data, batch_size, seq_length, device):
    starts = torch.randint(0, len(data) - seq_length - 1, (batch_size,))

    x = torch.stack([
        torch.from_numpy(np.asarray(data[i:i + seq_length], dtype=np.int64))
        for i in starts.tolist()
    ])

    y = torch.stack([
        torch.from_numpy(np.asarray(data[i + 1:i + seq_length + 1], dtype=np.int64))
        for i in starts.tolist()
    ])

    return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

In [ ]:
x, y = get_batch(train_data, 2, 128, device)

print("Input shape :", x.shape)
print("Target shape:", y.shape)

assert torch.equal(x[:, 1:], y[:, :-1])

print("✅ Virgo next-token batch loader passed")

del x
del y

torch.cuda.empty_cache()

In [ ]:
micro_batch_size = 8
seq_length = 1024
gradient_accumulation_steps = 16

learning_rate = 3e-4
min_learning_rate = 3e-5
weight_decay = 0.1

warmup_steps = 1000
max_steps = math.ceil(train_tokens / (micro_batch_size * seq_length * gradient_accumulation_steps))

eval_interval = 500
eval_steps = 20
checkpoint_interval = 1000

tokens_per_step = micro_batch_size * seq_length * gradient_accumulation_steps

print(f"Micro batch size       : {micro_batch_size}")
print(f"Sequence length        : {seq_length}")
print(f"Gradient accumulation  : {gradient_accumulation_steps}")
print(f"Tokens / optimizer step: {tokens_per_step:,}")
print(f"Maximum steps          : {max_steps:,}")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=weight_decay,
    fused=True
)

In [ ]:
def get_lr(step):
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps

    if step >= max_steps:
        return min_learning_rate

    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))

    return min_learning_rate + coefficient * (learning_rate - min_learning_rate)

In [ ]:
@torch.no_grad()
def estimate_val_loss():
    model.eval()

    losses = []

    for _ in range(eval_steps):
        x, y = get_batch(val_data, micro_batch_size, seq_length, device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

        losses.append(loss.item())

    model.train()

    return sum(losses) / len(losses)

In [ ]:
CHECKPOINT_DIR = "./virgo_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(step, tokens_seen, loss):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"virgo_step_{step}.pt")

    checkpoint = {
        "step": step,
        "tokens_seen": tokens_seen,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss,
        "config": {
            "vocab_size": vocab_size,
            "d_model": d_model,
            "num_heads": num_heads,
            "num_layers": num_layers,
            "d_ff": d_ff,
            "max_seq_length": max_seq_length,
            "dropout": dropout
        }
    }

    torch.save(checkpoint, checkpoint_path)

    print(f"💾 Checkpoint saved: {checkpoint_path}")

In [ ]:
torch.cuda.reset_peak_memory_stats()

print("Allocated VRAM:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved VRAM :", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

In [ ]:
model.train()

benchmark_steps = 10

optimizer.zero_grad(set_to_none=True)

start_time = time.time()
benchmark_tokens = 0

for step in range(benchmark_steps):
    for micro_step in range(gradient_accumulation_steps):
        x, y = get_batch(train_data, micro_batch_size, seq_length, device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
            loss = loss / gradient_accumulation_steps

        loss.backward()

        benchmark_tokens += x.numel()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

elapsed = time.time() - start_time
tokens_per_second = benchmark_tokens / elapsed
peak_vram = torch.cuda.max_memory_allocated() / 1024**3

print(f"Benchmark time : {elapsed:.2f}s")
print(f"Tokens/sec     : {tokens_per_second:,.0f}")
print(f"Peak VRAM      : {peak_vram:.2f} GB")
print(f"Final loss     : {loss.item() * gradient_accumulation_steps:.4f}")